# Protein Structure Visualization
Input an amino acid sequence and predict + visualize its 3D structure using **ESMFold** (Meta AI) and **py3Dmol**.

In [ ]:
import requests
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from biopandas.pdb import PandasPdb
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 1. Input your amino acid sequence

In [ ]:
# Enter your amino acid sequence here (single-letter codes)
# Default: lysozyme fragment
sequence = "KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL"

# Clean the sequence (remove spaces, newlines, make uppercase)
sequence = sequence.replace(" ", "").replace("\n", "").upper()
print(f"Sequence length: {len(sequence)} residues")
print(f"Sequence: {sequence}")

## 2. Predict 3D structure using ESMFold (Meta AI)

In [ ]:
def predict_structure(sequence: str, output_pdb: str = "predicted_structure.pdb") -> str:
    """Call ESMFold API to predict protein structure from sequence."""
    print("Sending sequence to ESMFold API...")
    url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    response = requests.post(url, data=sequence, headers=headers, timeout=120)

    if response.status_code != 200:
        raise RuntimeError(f"ESMFold API error {response.status_code}: {response.text}")

    pdb_string = response.text
    with open(output_pdb, "w") as f:
        f.write(pdb_string)
    print(f"Structure predicted and saved to '{output_pdb}'")
    return pdb_string


pdb_string = predict_structure(sequence)

## 3. Visualize 3D structure with py3Dmol

In [ ]:
def visualize_structure(pdb_string: str, style: str = "cartoon", color_scheme: str = "spectrum"):
    """Render protein structure interactively.

    Parameters
    ----------
    style        : 'cartoon', 'stick', 'sphere', 'surface'
    color_scheme : 'spectrum', 'chain', 'residue', 'b-factor'
    """
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_string, "pdb")

    color_map = {
        "spectrum": {"color": "spectrum"},
        "chain":    {"colorscheme": "chainHetatm"},
        "residue":  {"colorscheme": "amino"},
        "b-factor": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": 100}},
    }
    color_opts = color_map.get(color_scheme, {"color": "spectrum"})

    if style == "surface":
        view.addSurface(py3Dmol.VDW, {"opacity": 0.8, **color_opts})
    else:
        view.setStyle({style: color_opts})

    view.zoomTo()
    view.spin(True)
    view.show()


# --- Cartoon colored by N→C spectrum (blue→red) ---
visualize_structure(pdb_string, style="cartoon", color_scheme="spectrum")

In [ ]:
# --- Cartoon colored by amino acid type ---
visualize_structure(pdb_string, style="cartoon", color_scheme="residue")

In [ ]:
# --- pLDDT confidence score (stored in B-factor column by ESMFold) ---
# Blue = high confidence, Red = low confidence
visualize_structure(pdb_string, style="cartoon", color_scheme="b-factor")

## 4. Sequence & physicochemical analysis

In [ ]:
def plot_sequence_analysis(sequence: str):
    analysis = ProteinAnalysis(sequence)

    aa_counts = analysis.count_amino_acids()
    aa_percent = analysis.get_amino_acids_percent()
    mw = analysis.molecular_weight()
    ip = analysis.isoelectric_point()
    gravy = analysis.gravy()
    instability = analysis.instability_index()
    sec_struct = analysis.secondary_structure_fraction()  # (helix, turn, sheet)

    print("=" * 50)
    print("PROTEIN PHYSICOCHEMICAL PROPERTIES")
    print("=" * 50)
    print(f"  Length            : {len(sequence)} residues")
    print(f"  Molecular weight  : {mw:,.1f} Da")
    print(f"  Isoelectric point : {ip:.2f}")
    print(f"  GRAVY index       : {gravy:.3f}  ({'hydrophobic' if gravy > 0 else 'hydrophilic'})")
    print(f"  Instability index : {instability:.1f}  ({'unstable' if instability > 40 else 'stable'})")
    print(f"  2° structure est. : {sec_struct[0]*100:.1f}% helix | "
          f"{sec_struct[2]*100:.1f}% sheet | {sec_struct[1]*100:.1f}% turn")
    print("=" * 50)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Sequence Analysis", fontsize=14, fontweight="bold")

    # --- Amino acid composition ---
    aas = sorted(aa_percent, key=aa_percent.get, reverse=True)
    vals = [aa_percent[a] * 100 for a in aas]
    colors = plt.cm.tab20(np.linspace(0, 1, len(aas)))
    axes[0].bar(aas, vals, color=colors)
    axes[0].set_title("Amino Acid Composition (%)")
    axes[0].set_xlabel("Amino Acid")
    axes[0].set_ylabel("Percentage")
    axes[0].tick_params(axis="x", rotation=45)

    # --- Secondary structure estimate ---
    labels = ["α-Helix", "β-Sheet", "Turn"]
    sizes = [sec_struct[0], sec_struct[2], sec_struct[1]]
    pie_colors = ["#4C72B0", "#DD8452", "#55A868"]
    axes[1].pie(sizes, labels=labels, colors=pie_colors, autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Secondary Structure Estimate")

    # --- Hydrophobicity sliding window ---
    window = 9
    kd = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
          "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
          "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
          "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
    scores = [kd.get(aa, 0) for aa in sequence]
    hydro = [np.mean(scores[i:i+window]) for i in range(len(scores) - window + 1)]
    x = range(window // 2, len(hydro) + window // 2)
    axes[2].plot(x, hydro, color="steelblue", linewidth=1.5)
    axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.8)
    axes[2].fill_between(x, hydro, 0,
                         where=[h > 0 for h in hydro], alpha=0.3, color="orange", label="hydrophobic")
    axes[2].fill_between(x, hydro, 0,
                         where=[h < 0 for h in hydro], alpha=0.3, color="steelblue", label="hydrophilic")
    axes[2].set_title(f"Kyte-Doolittle Hydrophobicity (window={window})")
    axes[2].set_xlabel("Residue position")
    axes[2].set_ylabel("Hydrophobicity score")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig("sequence_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_sequence_analysis(sequence)

## 5. Per-residue pLDDT confidence (from ESMFold B-factor column)

In [ ]:
def plot_plddt(pdb_file: str = "predicted_structure.pdb"):
    ppdb = PandasPdb().read_pdb(pdb_file)
    atoms = ppdb.df["ATOM"]
    # One pLDDT score per residue (use CA atoms)
    ca = atoms[atoms["atom_name"] == "CA"][["residue_number", "b_factor"]]
    ca = ca.reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ["#0053D6" if v >= 90 else "#65CBF3" if v >= 70 else "#FFDB13" if v >= 50 else "#FF7D45"
              for v in ca["b_factor"]]
    ax.bar(ca["residue_number"], ca["b_factor"], color=colors, width=1.0)
    ax.axhline(90, color="#0053D6", linestyle="--", linewidth=0.8, label="Very high (≥90)")
    ax.axhline(70, color="#65CBF3", linestyle="--", linewidth=0.8, label="Confident (≥70)")
    ax.axhline(50, color="#FFDB13", linestyle="--", linewidth=0.8, label="Low (≥50)")
    ax.set_ylim(0, 100)
    ax.set_xlabel("Residue number")
    ax.set_ylabel("pLDDT score")
    ax.set_title("Per-residue pLDDT Confidence Score (ESMFold)")
    patches = [
        mpatches.Patch(color="#0053D6", label="Very high (≥90)"),
        mpatches.Patch(color="#65CBF3", label="Confident (70–90)"),
        mpatches.Patch(color="#FFDB13", label="Low (50–70)"),
        mpatches.Patch(color="#FF7D45", label="Very low (<50)"),
    ]
    ax.legend(handles=patches, loc="lower right")
    plt.tight_layout()
    plt.savefig("plddt_plot.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Mean pLDDT: {ca['b_factor'].mean():.1f}")


plot_plddt()